# Key Management Helpers

AES-256-GCM per-subject Data Encryption Key (DEK) management backed by
**Databricks secret scopes** as the Key Encryption Key (KEK) store.

> ⚠️ **WARNING — DEV / NON-PRODUCTION ONLY**
> DEKs are stored encrypted under a master key kept in a Databricks secret scope.
> This is NOT suitable for production PII. Replace with Azure Key Vault or AWS KMS
> before handling real customer data. There is intentionally no DEK recovery path
> after `shred_subject()` is called — that is the crypto-shredding guarantee.

## API
- `get_or_create_dek(subject_id, subject_type) -> bytes`
- `get_dek_map(subject_ids, subject_type) -> dict[str, bytes | None]`
- `encrypt_value(plaintext, dek) -> bytes`
- `decrypt_value(ciphertext, dek) -> str`  — returns `'[ERASED]'` when DEK is None
- `shred_subject(subject_id, subject_type)` — deletes secret + stamps `shredded_at`

## Configuration
- Secret scope: `dvdrental-dq` (must be created once in Databricks UI / CLI)
- KEK secret key: `master-kek`  (32-byte random, base64-encoded)
- DEK secret keys: `dek-{subject_type}-{subject_id}` (AES-256 key, base64-encoded)
- Key store table: `monitoring.subject_key_store`

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
SECRET_SCOPE = "dvdrental-dq"
KEK_SECRET   = "master-kek"        # 32-byte random key, base64-encoded, stored in scope
CATALOG      = "workspace"
KEY_STORE_FQN = f"{CATALOG}.monitoring.subject_key_store"

print(f"Secret scope : {SECRET_SCOPE}")
print(f"Key store    : {KEY_STORE_FQN}")

In [ ]:
import base64
import os
from datetime import datetime, timezone

from cryptography.hazmat.primitives.ciphers.aead import AESGCM


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _get_kek() -> bytes:
    """Fetch the master KEK from the Databricks secret scope."""
    raw = dbutils.secrets.get(scope=SECRET_SCOPE, key=KEK_SECRET)
    return base64.b64decode(raw)


def _dek_secret_key(subject_id: str, subject_type: str) -> str:
    return f"dek-{subject_type}-{subject_id}"


def _store_dek_metadata(subject_id: str, subject_type: str, encrypted_dek: bytes) -> None:
    """Upsert a row in subject_key_store to track DEK lifecycle."""
    from pyspark.sql import Row
    row = Row(
        subject_id    = str(subject_id),
        subject_type  = subject_type,
        encrypted_dek = encrypted_dek,
        kek_version   = "v1",
        created_at    = datetime.now(timezone.utc),
        shredded_at   = None,
    )
    df = spark.createDataFrame([row])
    from delta.tables import DeltaTable
    (
        DeltaTable.forName(spark, KEY_STORE_FQN)
        .alias("t")
        .merge(
            df.alias("s"),
            "t.subject_id = s.subject_id AND t.subject_type = s.subject_type",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [ ]:
# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def get_or_create_dek(subject_id, subject_type: str) -> bytes:
    """
    Return the AES-256 DEK for this subject, creating one if it does not exist.

    The DEK is stored in the Databricks secret scope encrypted under the KEK
    (AES-256-GCM). The subject_key_store table tracks metadata only.
    """
    secret_key = _dek_secret_key(str(subject_id), subject_type)
    try:
        raw = dbutils.secrets.get(scope=SECRET_SCOPE, key=secret_key)
        return base64.b64decode(raw)
    except Exception:
        # DEK does not exist — generate a fresh one
        dek = os.urandom(32)          # 256-bit AES key
        kek = _get_kek()

        # Encrypt DEK under KEK for metadata storage
        nonce = os.urandom(12)
        encrypted_dek = nonce + AESGCM(kek).encrypt(nonce, dek, None)

        # Persist plaintext DEK in secret scope (Databricks encrypts at rest)
        dbutils.secrets.put(scope=SECRET_SCOPE, key=secret_key, string_value=base64.b64encode(dek).decode())

        # Track in key store
        _store_dek_metadata(str(subject_id), subject_type, encrypted_dek)

        return dek


def get_dek_map(subject_ids, subject_type: str) -> dict:
    """
    Batch-fetch DEKs for a collection of subject IDs.

    Returns dict[str, bytes | None] — None means the subject has been shredded.
    Avoids N+1 secret calls by looking up all at once (one call per unique ID).
    """
    result = {}
    for sid in set(str(s) for s in subject_ids):
        secret_key = _dek_secret_key(sid, subject_type)
        try:
            raw = dbutils.secrets.get(scope=SECRET_SCOPE, key=secret_key)
            result[sid] = base64.b64decode(raw)
        except Exception:
            result[sid] = None   # shredded or never created
    return result


def encrypt_value(plaintext: str, dek: bytes) -> bytes:
    """
    Encrypt a UTF-8 string with AES-256-GCM.

    Output format: 12-byte nonce || ciphertext+tag (authenticated).
    """
    if plaintext is None:
        return None
    nonce = os.urandom(12)
    ct    = AESGCM(dek).encrypt(nonce, plaintext.encode("utf-8"), None)
    return nonce + ct


def decrypt_value(ciphertext: bytes, dek) -> str:
    """
    Decrypt an AES-256-GCM ciphertext produced by encrypt_value().

    Returns '[ERASED]' when dek is None (subject has been crypto-shredded).
    """
    if dek is None:
        return "[ERASED]"
    if ciphertext is None:
        return None
    nonce, ct = ciphertext[:12], ciphertext[12:]
    return AESGCM(dek).decrypt(nonce, ct, None).decode("utf-8")


def shred_subject(subject_id, subject_type: str) -> None:
    """
    Crypto-shred a subject: delete their DEK secret + stamp shredded_at.

    After this call, decrypt_value(..., None) returns '[ERASED]' for all
    previously encrypted columns for this subject.

    WARNING: This operation is IRREVERSIBLE. The DEK cannot be recovered.
    """
    secret_key = _dek_secret_key(str(subject_id), subject_type)
    try:
        dbutils.secrets.delete(scope=SECRET_SCOPE, key=secret_key)
    except Exception as exc:
        # Key may already be deleted (idempotent shred)
        print(f"Note: secret delete raised ({exc}) — treating as already shredded")

    # Stamp shredded_at in key store
    spark.sql(f"""
        UPDATE {KEY_STORE_FQN}
        SET shredded_at = current_timestamp()
        WHERE subject_id = '{str(subject_id)}'
          AND subject_type = '{subject_type}'
          AND shredded_at IS NULL
    """)
    print(f"Shredded DEK for {subject_type} subject_id={subject_id}")

In [ ]:
# ---------------------------------------------------------------------------
# Round-trip verification (run manually to smoke-test the setup)
# ---------------------------------------------------------------------------
# Uncomment to run:
#
# TEST_SUBJECT_ID   = 99999
# TEST_SUBJECT_TYPE = "customer"
# TEST_PLAINTEXT    = "test.user@example.com"
#
# dek = get_or_create_dek(TEST_SUBJECT_ID, TEST_SUBJECT_TYPE)
# ct  = encrypt_value(TEST_PLAINTEXT, dek)
# assert decrypt_value(ct, dek) == TEST_PLAINTEXT, "Round-trip FAILED"
# print("Round-trip PASS")
#
# shred_subject(TEST_SUBJECT_ID, TEST_SUBJECT_TYPE)
# assert decrypt_value(ct, get_dek_map([TEST_SUBJECT_ID], TEST_SUBJECT_TYPE).get(str(TEST_SUBJECT_ID))) == "[ERASED]"
# print("Post-shred PASS — returns [ERASED]")
print("NB_key_management_helpers loaded. Uncomment the verification block to smoke-test.")